# Data cleaning

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
df = pd.read_csv("../data/raw/books_raw.csv")
df.head()

,title,category,book_url,image_url,description,image_path
0,Soumission,Fiction,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/3e/ef/3...,"Dans une France assez proche de la nôtre, un h...",images/book_0.jpg
1,Private Paris (Private #10),Fiction,https://books.toscrape.com/catalogue/private-p...,https://books.toscrape.com/media/cache/9d/05/9...,Paris is burning--and only Private's Jack Morg...,images/book_1.jpg
2,"We Love You, Charlie Freeman",Fiction,https://books.toscrape.com/catalogue/we-love-y...,https://books.toscrape.com/media/cache/5f/15/5...,"The Freeman family--Charles, Laurel, and their...",images/book_2.jpg
3,Thirst,Fiction,https://books.toscrape.com/catalogue/thirst_94...,https://books.toscrape.com/media/cache/c4/0a/c...,"On a searing summer Friday, Eddie Chapman has ...",images/book_3.jpg
4,The Murder That Never Was (Forensic Instincts #5),Fiction,https://books.toscrape.com/catalogue/the-murde...,https://books.toscrape.com/media/cache/dc/44/d...,"Given the opportunity, would you assume someon...",images/book_4.jpg


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175 entries, 0 to 174
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        175 non-null    str  
 1   category     175 non-null    str  
 2   book_url     175 non-null    str  
 3   image_url    175 non-null    str  
 4   description  175 non-null    str  
 5   image_path   175 non-null    str  
dtypes: str(6)
memory usage: 8.3 KB


### Text Data Cleaning

#### Removing whitespace

In [5]:
df["title"] = df["title"].str.strip()
df["description"] = df["description"].str.strip()

In [6]:
df["title"] = df["title"].str.replace(r"\s+", " ", regex=True).str.strip()
df["description"] = df["description"].str.replace(r"\s+", " ", regex=True).str.strip()

In [7]:
print(df['description'].str.endswith('more').sum())

171


#### Removing irrelevant trailing word

In [8]:
print(df['description'].iloc[0])

Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, s Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, se développe comme un mauvais rêve.Le talent de l’auteur, sa force visionnaire nous entraînent sur un terrain ambigu et glissant ; son regard sur notre civilisation vieillissante fait coexister dans ce roman les intuitions poétiques, les effets comiq

In [9]:
df["description"] = df["description"].str.replace(r"\s*\.\.\.more$", "", regex=True)
print(df['description'].str.endswith('more').sum())

0


In [10]:
print(df['description'].iloc[0])

Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, s Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, se développe comme un mauvais rêve.Le talent de l’auteur, sa force visionnaire nous entraînent sur un terrain ambigu et glissant ; son regard sur notre civilisation vieillissante fait coexister dans ce roman les intuitions poétiques, les effets comiq

#### Removing invalid Data

In [11]:
same_describtion_title = df["description"].str.lower() == df["title"].str.lower()

print(df[same_describtion_title][["title", "category"]])

                              title    category
169  Kindle Paperwhite User's Guide  Nonfiction


In [12]:
df = df[(same_describtion_title == False)]

##### Removing datapoint that has french description

In [13]:
print(df[df["title"] == "Soumission"]["description"].iloc[0])

Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, s Dans une France assez proche de la nôtre, un homme s’engage dans la carrière universitaire. Peu motivé par l’enseignement, il s’attend à une vie ennuyeuse mais calme, protégée des grands drames historiques. Cependant les forces en jeu dans le pays ont fissuré le système politique jusqu’à provoquer son effondrement. Cette implosion sans soubresauts, sans vraie révolution, se développe comme un mauvais rêve.Le talent de l’auteur, sa force visionnaire nous entraînent sur un terrain ambigu et glissant ; son regard sur notre civilisation vieillissante fait coexister dans ce roman les intuitions poétiques, les effets comiq

In [14]:
df = df[df["title"] != "Soumission"]

In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 173 entries, 1 to 174
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   title        173 non-null    str  
 1   category     173 non-null    str  
 2   book_url     173 non-null    str  
 3   image_url    173 non-null    str  
 4   description  173 non-null    str  
 5   image_path   173 non-null    str  
dtypes: str(6)
memory usage: 9.5 KB


## remove repeated text from description

In [16]:
def has_repeated_text(text):
    text = str(text).strip()

    prefix = text[:80]
    repeated_at = text.find(prefix, 80)

    return 250 <= repeated_at <= 500


repeated_count = df["description"].apply(has_repeated_text).sum()
total_count = len(df)

print(f"number of descriptions with repeated text: {repeated_count}")

number of descriptions with repeated text: 168


In [17]:
def remove_duplicate_text(text):
    text = text.strip()

    prefix = text[:80]
    repeated_at = text.find(prefix, 80)

    if 250 <= repeated_at <= 500:
        text = text[repeated_at:]

    return text.strip()

df["description"] = df["description"].apply(remove_duplicate_text)

### Saving new cleaned Data


In [18]:
from pathlib import Path

clean_folder = Path("../data/cleaned")
clean_folder.mkdir(parents=True, exist_ok=True)

df.to_csv(clean_folder / "books_cleaned.csv", index=False)

In [19]:
print("Rows saved:", len(df))
print(df["category"].value_counts())


Rows saved: 173
category
Nonfiction    109
Fiction        64
Name: count, dtype: int64
